# Type2 HFSS Setup-Ready

이 노트북은 code-owned official type2 setup-ready helper를 호출하는 thin manual notebook이다. `entry.setup_type2_step.export_and_setup_type2_step_into_hfss(...)`가 canonical type2 export + existing AEDT desktop attach setup-ready runtime을 함께 소유하고, notebook은 그 결과 `.aedt`와 import handoff ledger를 확인만 한다.

- owner helper: `entry.setup_type2_step.export_and_setup_type2_step_into_hfss(...)`
- GUI attach runtime: `Hfss(..., non_graphical=False, new_desktop=False)`
- canonical scene STEP: `run/step/type2/type2_scene.step`
- 결과물: `run/aedt/type2_step_setup_ready/type2_setup_ready.aedt`, `run/aedt/type2_step_import/type2_imported_ledger.json`
- import-only ledger는 ownership handoff artifact이고, mesh/boundary/ports/analysis는 setup-ready runtime이 owner다.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint
import subprocess
import sys

repo_root_result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    check=True,
    capture_output=True,
    text=True,
)
repo_root_text = repo_root_result.stdout.strip()
if repo_root_text == "":
    raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
REPO_ROOT = Path(repo_root_text).resolve()
pyproject_path = REPO_ROOT / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from entry.setup_type2_step import export_and_setup_type2_step_into_hfss
from peetsfea.aedt import Hfss

TYPE2_TOML_PATH = REPO_ROOT / "examples" / "type2_fixed.toml"
STEP_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
STEP_LEDGER_PATH = STEP_OUTPUT_DIR / "type2_step_ledger.json"
TYPE2_SCENE_STEP_PATH = STEP_OUTPUT_DIR / "type2_scene.step"
OUTPUT_AEDT_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_setup_ready" / "type2_setup_ready.aedt"
IMPORTED_LEDGER_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_import" / "type2_imported_ledger.json"
DESIGN_NAME = "type2_step_setup_ready"


## 1. Run Official Attached Setup-Ready Pipeline

이 셀은 notebook 안에서 export/import/setup를 따로 orchestration하지 않는다. `export_and_setup_type2_step_into_hfss(...)`를 호출해 canonical type2 export + existing AEDT desktop attach setup-ready runtime을 한 번에 수행한다.


In [2]:
setup_result = export_and_setup_type2_step_into_hfss(
    hfss=Hfss(project=None, design=DESIGN_NAME, non_graphical=False, new_desktop=False),
    toml_path=TYPE2_TOML_PATH,
    output_dir=STEP_OUTPUT_DIR,
    step_ledger_path=STEP_LEDGER_PATH,
    output_aedt_path=OUTPUT_AEDT_PATH,
    imported_ledger_path=IMPORTED_LEDGER_PATH,
    seed=0,
)

print(f"AEDT path: {setup_result['aedt_path']}")
print(f"imported ledger: {setup_result['imported_ledger_path']}")
print(
    "mesh: "
    f"{setup_result['mesh']['operation_name']} "
    f"objects={setup_result['mesh']['objects']} "
    f"max_length={setup_result['mesh']['max_length']}"
)
print(
    "boundary: "
    f"{setup_result['boundary']['type']} region={setup_result['boundary']['region_name']} "
    f"faces={setup_result['boundary']['face_count']} "
    f"offset={setup_result['boundary']['offset_value']}"
)
print(f"ports: tx={setup_result['ports']['tx']} rx={setup_result['ports']['rx']}")
print(f"validation: {setup_result['validation_report']}")
print("Runtime detached the notebook HFSS handle after setup/save.")


PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.25.1.
PyAEDT INFO: Initializing Desktop session.
PyAEDT INFO: AEDT version 2025.2.
PyAEDT INFO: New AEDT session is starting on gRPC port 38693.
PyAEDT INFO: Starting new AEDT gRPC session on port 38693.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: TransportMode.UDS
PyAEDT INFO: Electronics Desktop started on gRPC port 38693 after 10.8 seconds.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 38693.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Project Project66 has been created.
PyAEDT INFO: Added design 'type2_step_setup_ready' of type HFSS.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler

RuntimeError: PyAEDT materials lookup did not resolve ferrite material after project definition sync (material_name=MULL12060ferrite)

## 2. Inspect Import Handoff Ledger

이 셀은 runtime이 쓴 imported ledger를 읽어서 import-only handoff를 요약한다. notebook은 import semantics, setup-ready mesh/boundary/port logic를 재구현하지 않는다.


In [ ]:
payload = json.loads(IMPORTED_LEDGER_PATH.read_text(encoding="utf-8"))

summary = {
    "aedt_path": payload["aedt_path"],
    "source_step_ledger_path": payload["source_step_ledger_path"],
    "scene_step_path": payload["scene_step_path"],
    "non_model_count": len(payload["non_model_objects"]),
    "modeled_count": len(payload["modeled_objects"]),
}
pprint(summary)

for group_name in ("non_model_objects", "modeled_objects"):
    print()
    print(group_name)
    print("-" * len(group_name))
    for entry in payload[group_name]:
        print(f"object_id: {entry['object_id']}")
        print(f"  role: {entry['role']}")
        print(f"  model_state: {entry['model_state']}")
        print(f"  imported_object_names: {entry['imported_object_names']}")
